# Project 4: Image/Text Recognition Pipeline
### Path 1 — OCR (pytesseract + OpenCV) — with Gradio frontend

**Gatekeeper Rule (must pass all 4):**
1. Library Integration — pytesseract, error-free
2. Pre-Processing Integrity — Grayscale + Adaptive Thresholding
3. Accuracy Benchmarking — min 80% confidence
4. Visual Confirmation — clean OCR text output

**v3 update:** structured documents (ID cards) with security-pattern backgrounds were producing gibberish (background pattern misread as text). Fix: added denoising-based variants (Non-local-means, Bilateral filter) that strip pattern noise while keeping text edges, plus per-word confidence filtering to drop garbage tokens.

Run cells top to bottom in Colab. Last cell launches the Gradio UI (public share link included).

**Privacy note:** `share=True` creates a public tunnel link — anyone with it can upload/see images processed. Don't run real ID documents through it; use a sample/redacted image for testing.

In [ ]:
# 1. Install system + python deps (Colab)
!apt-get -qq update && apt-get -qq install -y tesseract-ocr
!pip install -q pytesseract opencv-python-headless gradio pillow numpy

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# 2. Imports
import cv2
import numpy as np
import pytesseract
import gradio as gr

MIN_WORD_CONF = 35  # drop garbage tokens below this confidence

In [ ]:
# 3. Pre-Processing: multiple variants (Gatekeeper #2)
# v1: adaptive/Otsu thresholds (thumbnails, stylized text)
# v3: + denoising variants (Non-local-means, Bilateral) for structured docs / security-pattern backgrounds
def generate_preprocess_variants(image):
    img = np.array(image)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img

    # Upscale 2x -> helps small/dense text
    gray_up = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # Contrast boost (CLAHE)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_clahe = clahe.apply(gray_up)

    variants = {}

    # -- Path A: light Gaussian blur (good for clean/stylized text) --
    blur = cv2.GaussianBlur(gray_clahe, (3, 3), 0)
    variants['adaptive'] = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 11)
    variants['adaptive_inv'] = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 11)
    _, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants['otsu'] = otsu
    _, otsu_inv = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    variants['otsu_inv'] = otsu_inv

    # -- Path B: Non-local-means denoise (kills fine security-pattern/watermark noise) --
    denoised = cv2.fastNlMeansDenoising(gray_clahe, h=15)
    _, den_otsu = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants['denoised_otsu'] = den_otsu
    _, den_otsu_inv = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    variants['denoised_otsu_inv'] = den_otsu_inv

    # -- Path C: Bilateral filter (smooths background, keeps text edges sharp) --
    bilateral = cv2.bilateralFilter(gray_clahe, 9, 75, 75)
    _, bil_otsu = cv2.threshold(bilateral, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants['bilateral_otsu'] = bil_otsu
    _, bil_otsu_inv = cv2.threshold(bilateral, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    variants['bilateral_otsu_inv'] = bil_otsu_inv

    return gray, variants

In [ ]:
# 4. OCR + auto-pick best variant/PSM (Gatekeeper #1 + #3)
def extract_text_with_confidence(processed_img, psm=6):
    config = f'--oem 3 --psm {psm}'
    data = pytesseract.image_to_data(processed_img, config=config, output_type=pytesseract.Output.DICT)

    words, confs = [], []
    for i in range(len(data['text'])):
        w = data['text'][i].strip()
        c = int(data['conf'][i])
        if w and c >= MIN_WORD_CONF:   # drop low-confidence garbage tokens
            words.append(w)
            confs.append(c)

    full_text = " ".join(words)
    avg_conf = round(sum(confs) / len(confs), 2) if confs else 0.0
    return full_text, avg_conf


def best_ocr_result(variants):
    """Try each pre-process variant x each PSM mode, keep highest-confidence result."""
    best = {'name': 'none', 'text': '', 'conf': -1.0, 'img': None}
    for psm in (6, 11):  # 6 = uniform block (documents), 11 = sparse text (thumbnails/banners)
        for name, img in variants.items():
            text, conf = extract_text_with_confidence(img, psm=psm)
            if conf > best['conf']:
                best.update(name=f"{name} / psm{psm}", text=text, conf=conf, img=img)
    return best

In [ ]:
# 5. Gatekeeper Validation (all 4 checks)
def gatekeeper_report(full_text, avg_conf, thresh_img, winning_variant):
    checks = {
        "1. Library Integration (pytesseract)": True,
        "2. Pre-Processing Integrity (Grayscale+AdaptiveThreshold)": thresh_img is not None,
        "3. Accuracy Benchmarking (>=80% confidence)": avg_conf >= 80,
        "4. Visual Confirmation (non-empty text output)": len(full_text.strip()) > 0,
    }
    lines = [f"[{'PASS' if passed else 'FAIL'}] {name}" for name, passed in checks.items()]
    overall = "ALL CHECKS PASSED" if all(checks.values()) else "MILESTONE NOT MET"
    lines.append("")
    lines.append(f"Winning pre-process/PSM combo: {winning_variant}")
    lines.append(overall)
    return "\n".join(lines)

In [ ]:
# 6. Full Pipeline (used by Gradio frontend)
def run_pipeline(image):
    if image is None:
        return None, "No image provided.", 0.0, "No image provided."

    gray, variants = generate_preprocess_variants(image)
    best = best_ocr_result(variants)
    report = gatekeeper_report(best['text'], best['conf'], best['img'], best['name'])

    display_img = cv2.cvtColor(best['img'], cv2.COLOR_GRAY2RGB)

    return display_img, best['text'] if best['text'] else "(no text detected)", best['conf'], report

In [ ]:
import gradio as gr

# 7. Gradio Frontend
demo = gr.Interface(
    fn=run_pipeline,
    inputs=gr.Image(type="pil", label="Upload Image (with text)"),
    outputs=[
        gr.Image(label="Pre-Processed Output (best variant auto-selected)"),
        gr.Textbox(label="Extracted Text (OCR)"),
        gr.Number(label="Average Confidence (%)"),
        gr.Textbox(label="Gatekeeper Validation Report", lines=7),
    ],
    title="Project 4: Image/Text Recognition Pipeline (OCR)",
    description="Upload an image containing text. Pipeline: multi-variant Grayscale/Denoise/Threshold -> Tesseract OCR (multi-PSM) -> best result auto-picked by confidence -> Gatekeeper Validation (min 80%)."
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ccf5e8f33e7f450d4b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
